# Week 30, Day 2: super()

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w030d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Book:
    def __init__(self, title, author):
        print("Book.__init__ runs")
        self.title = title
        self.author = author

class AudioBook(Book):
    def __init__(self, title, author, minutes):
        print("AudioBook.__init__ starts")
        super().__init__(title, author)
        self.minutes = minutes
        print("AudioBook.__init__ ends")

a = AudioBook("Girl, Woman, Other", "Bernardine Evaristo", 700)
print(a.title, a.minutes)

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("opening balance can't be negative")
        self.owner = owner
        self.balance = balance

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, {self.balance})"

class SavingsAccount(Account):
    def __init__(self, owner, balance=0, rate=2):
        super().__init__(owner, balance)       # the parent checks and sets owner and balance
        self.rate = rate                        # the child's own part

s = SavingsAccount("Ngozi", 50000, rate=3)
print(s, s.rate)
try:
    SavingsAccount("Tom", -5)
except ValueError as err:
    print("ValueError:", err)

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

class ChildAccount(Account):
    """A child's account: always opened with a £10 gift from the bank."""

    def __init__(self, owner, guardian):
        super().__init__(owner, balance=1000)
        self.guardian = guardian

c = ChildAccount("Ada", guardian="Mrs Okafor")
print(c.owner, c.balance, c.guardian)

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError(f"only {self.balance} available")
        self.balance -= amount
        return self.balance

class LimitedAccount(Account):
    """At most 3 withdrawals."""

    def __init__(self, owner, balance=0):
        super().__init__(owner, balance)
        self.withdrawals = 0

    def withdraw(self, amount):
        if self.withdrawals >= 3:                       # before: an extra check
            raise ValueError("no withdrawals left")
        left = super().withdraw(amount)                 # the parent's rules and work
        self.withdrawals += 1                           # after: only if that succeeded
        return left

acc = LimitedAccount("Femi", 1000)
for amount in [100, 5000, 200, 300, 50]:
    try:
        print("left", acc.withdraw(amount))
    except ValueError as err:
        print("refused:", err)
print(acc.withdrawals)

In [ ]:
class Vehicle:
    def __init__(self, reg):
        self.reg = reg

    def describe(self):
        return self.reg

class Bus(Vehicle):
    def __init__(self, reg, route):
        super().__init__(reg)
        self.route = route

    def describe(self):
        return f"{super().describe()} on route {self.route}"

print(Vehicle("LA21 KDE").describe())
print(Bus("LX70 BUS", "Oshodi to Ikeja").describe())

**Predict before you run:** what will this print?



In [ ]:
class Item:
    def __init__(self, title):
        print("Item")
        self.title = title

    def label(self):
        return self.title

class Book(Item):
    def __init__(self, title, author):
        print("Book")
        super().__init__(title)
        self.author = author

    def label(self):
        return super().label() + " by " + self.author

class Ebook(Book):
    def __init__(self, title, author, mb):
        print("Ebook")
        super().__init__(title, author)
        self.mb = mb

    def label(self):
        return super().label() + f" ({self.mb} MB)"

e = Ebook("Aké", "Wole Soyinka", 3)
print(e.label())

In [ ]:
class LibraryError(Exception):
    """Something went wrong in the library."""

class NotOnShelf(LibraryError):
    def __init__(self, title, back_on):
        super().__init__(f"{title!r} is out until day {back_on}")
        self.title = title
        self.back_on = back_on

try:
    raise NotOnShelf("Small Island", 214)
except LibraryError as err:
    print(err)
    print(err.title, err.back_on, err.args)

In [ ]:
class Vehicle:
    def __init__(self, reg):
        self.reg = reg

    def describe(self):
        return self.reg

class Lorry(Vehicle):
    def __init__(self, reg, tonnes):
        super().__init__(reg)
        self.tonnes = tonnes

    def describe(self):
        return f"{self.describe()}, {self.tonnes} tonnes"     # should be super()

print(Lorry("KJA 123 AB", 18).describe())

> **Key idea.** `super().method(...)` runs the parent's version of a method on the same `self`. In `__init__`, pass the parent its share with `super().__init__(...)` first, then set the child's own attributes; in other methods, do checks before the call and records or extra results after it, and return what you mean to return. Inherit when there's nothing to change, replace when the parent's version is wrong for the child, extend with `super()` when it's right but incomplete. Exceptions can carry data: call `super().__init__(message)` and store the details as attributes.

### Exercise: An audiobook

Write `AudioBook`, a kind of `Book` that also takes `minutes`. Use `super().__init__` for the title and author. Add `hours()`, returning the length in hours rounded to 1 decimal place, and extend `label` so an audiobook's label ends with `, 11.7 hours` (one decimal place, always).

In [ ]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def label(self):
        return f"{self.title} by {self.author}"

class AudioBook(Book):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_audio():\n    \"an audiobook\"\n    A, B = ns[\"AudioBook\"], ns[\"Book\"]\n    a = A(\"Girl, Woman, Other\", \"Bernardine Evaristo\", 700)\n    assert issubclass(A, B) and (a.title, a.author, a.minutes) == (\"Girl, Woman, Other\", \"Bernardine Evaristo\", 700)\n    assert a.hours() == 11.7 and A(\"x\", \"y\", 60).hours() == 1.0\n    assert a.label() == \"Girl, Woman, Other by Bernardine Evaristo, 11.7 hours\", f\"got {a.label()!r}\"\n    assert A(\"Ak\u00e9\", \"Wole Soyinka\", 540).label() == \"Ak\u00e9 by Wole Soyinka, 9.0 hours\"\n    assert B(\"Ak\u00e9\", \"Wole Soyinka\").label() == \"Ak\u00e9 by Wole Soyinka\"\ndef test_super():\n    \"uses super()\"\n    assert source.count(\"super()\") >= 2, \"use super() in both __init__ and label\"")

### Exercise: An error with details

Write `OverBudget`, a kind of `ValueError` made as `OverBudget(item, over)`. Its message is `"rice is 250p over budget"` (with the real item and amount), and it keeps `item` and `over` as attributes. Then write `check(spending, budget)`: given a dictionary from item to pence spent and a dictionary of budgets, raise `OverBudget` for the **first** item (in `spending`'s order) that's over its budget, and return `None` otherwise. Items with no budget are fine.

In [ ]:
class OverBudget(ValueError):
    ___

def check(spending, budget):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_error():\n    \"the exception\"\n    O = ns[\"OverBudget\"]\n    e = O(\"rice\", 250)\n    assert issubclass(O, ValueError) and str(e) == \"rice is 250p over budget\", f\"str gave {str(e)!r}\"\n    assert e.item == \"rice\" and e.over == 250\ndef test_check():\n    \"checking\"\n    O = ns[\"OverBudget\"]\n    assert check({\"rice\": 900, \"oil\": 500}, {\"rice\": 1000, \"oil\": 500}) is None\n    assert check({\"bread\": 300}, {}) is None and check({}, {\"rice\": 1}) is None\n    try:\n        check({\"tea\": 100, \"rice\": 1250, \"oil\": 900}, {\"rice\": 1000, \"oil\": 600})\n        assert False, \"rice is over budget\"\n    except O as err:\n        assert (err.item, err.over) == (\"rice\", 250), f\"got {err.item}, {err.over}\"")

### Exercise: Three levels

Build the family `Vehicle(reg)`, `Car(reg, seats)` (a kind of `Vehicle`) and `Taxi(reg, seats, licence)` (a kind of `Car`). Each `__init__` passes the parent's share to `super().__init__`. Each `describe` extends its parent's:

- `Vehicle("AB12 CDE").describe()` gives `"AB12 CDE"`;
- `Car("AB12 CDE", 5).describe()` gives `"AB12 CDE, 5 seats"`;
- `Taxi("AB12 CDE", 5, "L4471").describe()` gives `"AB12 CDE, 5 seats, licence L4471"`.

A `Car` with fewer than 2 seats raises `ValueError` (so a `Taxi` does too, without repeating the check).

In [ ]:
class Vehicle:
    ___

class Car(Vehicle):
    ___

class Taxi(Car):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_describe():\n    \"descriptions\"\n    V, C, T = ns[\"Vehicle\"], ns[\"Car\"], ns[\"Taxi\"]\n    assert V(\"AB12 CDE\").describe() == \"AB12 CDE\"\n    assert C(\"AB12 CDE\", 5).describe() == \"AB12 CDE, 5 seats\"\n    t = T(\"AB12 CDE\", 5, \"L4471\")\n    assert t.describe() == \"AB12 CDE, 5 seats, licence L4471\", f\"got {t.describe()!r}\"\n    assert (t.reg, t.seats, t.licence) == (\"AB12 CDE\", 5, \"L4471\")\ndef test_family():\n    \"family and checks\"\n    V, C, T = ns[\"Vehicle\"], ns[\"Car\"], ns[\"Taxi\"]\n    assert issubclass(C, V) and issubclass(T, C)\n    for make in [lambda: C(\"X\", 1), lambda: T(\"X\", 0, \"L1\")]:\n        try:\n            make()\n            assert False, \"fewer than 2 seats should raise ValueError\"\n        except ValueError:\n            pass\n    assert source.count(\"super()\") >= 4, \"use super() in the child classes' __init__ and describe\"")

### Exercise: A basket with limits

`Basket` is given. Write `LimitedBasket(limit)`, a kind of `Basket` that allows at most `limit` of any one item (as on a sale day). Its `add` raises `ValueError` if adding would take that item over the limit, and otherwise lets `Basket.add` do the work, including its own check on `qty`. A refused `add` leaves the basket unchanged. `count` is inherited.

In [ ]:
class Basket:
    def __init__(self):
        self.items = {}

    def add(self, name, qty=1):
        if qty < 1:
            raise ValueError("qty must be at least 1")
        self.items[name] = self.items.get(name, 0) + qty

    def count(self):
        return sum(self.items.values())

class LimitedBasket(Basket):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_limits():\n    \"limits\"\n    L = ns[\"LimitedBasket\"]\n    b = L(2)\n    b.add(\"eggs\")\n    b.add(\"eggs\")\n    b.add(\"milk\", 2)\n    assert b.items == {\"eggs\": 2, \"milk\": 2} and b.count() == 4 and b.limit == 2\n    for name, qty in [(\"eggs\", 1), (\"milk\", 1), (\"tea\", 3)]:\n        try:\n            b.add(name, qty)\n            assert False, f\"adding {qty} {name} should go over the limit\"\n        except ValueError:\n            pass\n    try:\n        b.add(\"tea\", 0)\n        assert False, \"qty 0 should still be refused\"\n    except ValueError:\n        pass\n    assert b.items == {\"eggs\": 2, \"milk\": 2}, \"a refused add changed the basket\"\ndef test_inherited():\n    \"inherits from Basket\"\n    B, L = ns[\"Basket\"], ns[\"LimitedBasket\"]\n    assert issubclass(L, B) and \"count\" not in vars(L)\n    assert \"super()\" in source, \"use super()\"\n    b = B()\n    b.add(\"eggs\", 50)\n    assert b.count() == 50")

### Bug hunt: No balance

Opening a joint account crashes with `AttributeError: 'JointAccount' object has no attribute 'balance'`. Fix `JointAccount`.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def summary(self):
        return f"{self.owner}: {self.balance}"

class JointAccount(Account):
    def __init__(self, owner, second_owner, balance=0):
        self.second_owner = second_owner

    def summary(self):
        return f"{super().summary()} (with {self.second_owner})"

print(JointAccount("Bisi", "Kunle", 2500).summary())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the summary\"\n    expect_output(\"Bisi: 2500 (with Kunle)\")")

### Bug hunt: The missing price

A first-class ticket should print `Leeds to York: £14.00 (first class)` but prints `None`. Fix it.

In [ ]:
class Ticket:
    def __init__(self, origin, destination, pence):
        self.origin = origin
        self.destination = destination
        self.pence = pence

    def describe(self):
        return f"{self.origin} to {self.destination}: £{self.pence / 100:.2f}"

class FirstClass(Ticket):
    def describe(self):
        text = super().describe()
        text + " (first class)"

print(FirstClass("Leeds", "York", 1400).describe())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"first class\"\n    expect_output(\"Leeds to York: \u00a314.00 (first class)\")")

### Bug hunt: Forever

This crashes with `RecursionError: maximum recursion depth exceeded`. It should print `Jollof rice (serves 6, vegetarian)`. Fix it.

In [ ]:
class Recipe:
    def __init__(self, name, serves):
        self.name = name
        self.serves = serves

    def title(self):
        return f"{self.name} (serves {self.serves})"

class VegetarianRecipe(Recipe):
    def title(self):
        return self.title()[:-1] + ", vegetarian)"

print(VegetarianRecipe("Jollof rice", 6).title())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the title\"\n    expect_output(\"Jollof rice (serves 6, vegetarian)\")")

### Puzzle: A member with a card

Arrange the lines so that `Member` extends `Person` with a card number, using `super()`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    def __init__(self):
        self.log = ["A"]

    def f(self):
        return 1

class B(A):
    def __init__(self):
        super().__init__()
        self.log.append("B")

    def f(self):
        return super().f() + 10

class C(B):
    def __init__(self):
        self.log = ["C"]

    def f(self):
        return super().f() * 2

p = ___   # B().log
q = ___   # C().log
r = ___   # B().f()
s = ___   # C().f()
t = ___   # A().f()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"A\", \"B\"], \"p\"\n    assert q == [\"C\"], \"q\"\n    assert (r, s, t) == (11, 22, 1)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: accounts with overdrafts

Build a small family of bank accounts. Amounts are whole numbers of pence.

- `AccountError(Exception)`, and `InsufficientFunds(AccountError)` made as `InsufficientFunds(needed)`, with the message `"need 250 more"` and the attribute `needed`.
- `Account(owner, balance=0)`. `available()` returns how much can be withdrawn (here, the balance). `withdraw(amount)` raises `ValueError` if `amount` is less than 1, raises `InsufficientFunds` (with how much more would be needed) if `amount` is more than `available()`, and otherwise takes it off the balance, records `("out", amount)` in a list `history`, and returns the new balance. `deposit(amount)` raises `ValueError` if `amount` is less than 1, otherwise adds it, records `("in", amount)` and returns the new balance. `repr` gives `Account('Ngozi', 5000)` (the real class name for subclasses).
- `OverdraftAccount(owner, balance=0, overdraft=0)`, a kind of `Account`: the balance may go down to `-overdraft`. Extend `available()` with `super()`; don't rewrite `withdraw`.
- `StudentAccount(owner, balance=0)`, a kind of `OverdraftAccount` with a fixed overdraft of 50000 (£500). Its `withdraw` extends the parent's: withdrawals of more than 20000 at once raise `ValueError` (checked before anything else).

`repr` doesn't show the overdraft.

In [ ]:
class AccountError(Exception):
    pass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_account():\n    \"a plain account\"\n    A, IF = ns[\"Account\"], ns[\"InsufficientFunds\"]\n    a = A(\"Ngozi\", 5000)\n    assert a.withdraw(2000) == 3000 and a.deposit(500) == 3500 and a.history == [(\"out\", 2000), (\"in\", 500)]\n    assert repr(a) == \"Account('Ngozi', 3500)\" and a.available() == 3500\n    try:\n        a.withdraw(3750)\n        assert False, \"should raise InsufficientFunds\"\n    except IF as err:\n        assert str(err) == \"need 250 more\" and err.needed == 250, f\"got {str(err)!r}\"\n    for bad in [lambda: a.withdraw(0), lambda: a.deposit(-5)]:\n        try:\n            bad()\n            assert False, \"amounts below 1 should raise ValueError\"\n        except ValueError:\n            pass\n    assert a.balance == 3500 and len(a.history) == 2, \"a refused transaction changed the account\"\n    assert issubclass(IF, ns[\"AccountError\"]) and A(\"x\").balance == 0\ndef test_overdraft():\n    \"overdrafts\"\n    O, IF = ns[\"OverdraftAccount\"], ns[\"InsufficientFunds\"]\n    o = O(\"Tom\", 1000, overdraft=5000)\n    assert o.available() == 6000 and o.withdraw(4000) == -3000 and o.available() == 2000\n    try:\n        o.withdraw(2001)\n        assert False, \"only 2000 is available\"\n    except IF as err:\n        assert err.needed == 1, f\"needed should be 1, not {err.needed}\"\n    assert o.withdraw(2000) == -5000 and repr(o) == \"OverdraftAccount('Tom', -5000)\"\n    assert \"withdraw\" not in vars(O), \"OverdraftAccount should inherit withdraw\"\n    assert O(\"z\").available() == 0\ndef test_student():\n    \"student accounts\"\n    S, O, IF = ns[\"StudentAccount\"], ns[\"OverdraftAccount\"], ns[\"InsufficientFunds\"]\n    s = S(\"Ada\")\n    assert issubclass(S, O) and s.overdraft == 50000 and s.available() == 50000\n    try:\n        s.withdraw(20001)\n        assert False, \"more than 20000 at once should raise ValueError\"\n    except IF:\n        assert False, \"the 20000 limit is checked first, with ValueError\"\n    except ValueError:\n        pass\n    assert s.withdraw(20000) == -20000 and s.withdraw(20000) == -40000\n    try:\n        s.withdraw(15000)\n        assert False, \"only 10000 is left\"\n    except IF as err:\n        assert err.needed == 5000\n    assert s.history == [(\"out\", 20000), (\"out\", 20000)] and repr(s) == \"StudentAccount('Ada', -40000)\"")

## Warm-up

Inside `AudioBook.__init__`, the line `super().__init__(title, author)` runs **`Book`'s** `__init__` on the same object, so the title and author are set exactly as for a plain book. Then the child adds the one thing that's new. That's the whole idea of today: the child does its own part and hands the rest to the parent.

## Extending __init__

The pattern is nearly always the same: take all the arguments, pass the parent's share to `super().__init__(...)`, then set the child's own attributes.

- The parent's checks run for the child too, without being copied. If `Account` later checks something new, savings accounts get it for free.
- Call `super().__init__` **first**, before the child's own lines, unless you have a reason not to. Then the child's code can rely on the parent's attributes existing.
- The child decides its own signature. It can take extra arguments (`rate`), fill some in itself, or take fewer:

If the child has nothing to add, don't write an `__init__` at all: it's inherited, as on Day 1. A child `__init__` that only calls `super().__init__` with the same arguments is noise.

## Extending other methods

`super()` works for any method, not just `__init__`. A child's method can do something **before** the parent's version (usually a check), something **after** it (usually a record or a change to the result), or both:

Notice the order in the child's `withdraw`. The count goes up **after** `super().withdraw` returns, so the refused 5000 (not enough money) doesn't use up one of the three, and it's the last request, 50, that meets the limit. Putting `self.withdrawals += 1` first would quietly count failures too.

Extending also works on results. A child's `describe` can take the parent's text and add to it:

Compare the three choices a child has for a method:

| The child... | What it writes | Example |
|---|---|---|
| inherits it | nothing | `Ebook` keeps `label` |
| replaces it | a method with the same name, no `super()` | `ReferenceBook.label` (Day 1) |
| extends it | a method with the same name that calls `super()` | `LimitedAccount.withdraw` |

## What super() does

`super()` returns a stand-in for the same object that **starts its search for methods one step further up the family** than the class whose code is running. So inside `Bus.describe`, `super().describe()` skips `Bus` and finds `Vehicle.describe`, and calls it with the same `self`.

Because `self` doesn't change, the parent's method still sees the child's attributes and the child's versions of other methods. In a family of three, each level can extend the one above:

The calls go down the family (`Ebook`, then `Book`, then `Item`) and the work comes back up: `Item` gives the title, `Book` adds the author, `Ebook` adds the size.

You'll also see older code that calls the parent by name: `Book.__init__(self, title, author)`. It works here, but it writes the parent's name into the child (change the parent and you must find every such line) and it goes wrong with multiple parents, which Day 3 explains. Use `super()`.

## Exceptions that carry data

Yesterday's exceptions only had a message. A caller often wants the details too: which item, how much money was missing. Give the exception its own `__init__`, pass the message up with `super().__init__(message)`, and keep the details as attributes:

`str(err)` gives the message, because `Exception.__init__` stores it in `err.args`. Without `super().__init__(message)`, the error would print as an empty line, which is confusing when it reaches a traceback.

## Three mistakes

These three cause most `super()` trouble. Each runs the example below with one thing wrong:

| Mistake | What happens |
|---|---|
| The child's `__init__` forgets `super().__init__(...)` | the parent's attributes are missing: `AttributeError` later, far from the cause |
| `return` left off: `super().describe() + "!"` computed but not returned | the method returns `None` |
| `self.describe()` instead of `super().describe()` | the method calls **itself** forever: `RecursionError` |

Two more small ones show up as `TypeError`s: writing `super().__init__(self, reg)` passes `self` twice (`super()` already supplies it), and writing `super.__init__(reg)` without the brackets after `super` doesn't call `super()` at all.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a pair of classes you've written (in this course or your own code) where one repeats most of the other's `__init__`, and rewrite the second as a child that calls `super().__init__`. Tomorrow: overriding in more depth, classes with more than one parent, and the method resolution order that decides which version runs.